In [1]:
import numpy as np
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd


# -------------------------
# 1. Check probability validity
# -------------------------
def s_nan(y_prob, eps=1e-3):
    if not np.all(np.isfinite(y_prob)):
        return 0

    row_sums = np.sum(y_prob, axis=1)
    if np.all(np.abs(row_sums - 1.0) <= eps):
        return 1
    return 0


# -------------------------
# 2. Majority + JSD score
# -------------------------
def s_maj_jsd(y_true, y_pred, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_true))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)
    true_dist = np.bincount(y_true, minlength=n_classes) / len(y_true)

    # Majority score
    f_max = np.max(pred_dist)
    s_maj = 1 - (f_max - 1/n_classes) / (1 - 1/n_classes)

    # Jensen-Shannon divergence
    # jensenshannon() trả về căn bậc hai của JSD dùng log e
    js_distance = jensenshannon(true_dist, pred_dist)
    
    # Bình phương để lấy JSD (Divergence)
    js_divergence = js_distance**2
    s_jsd = 1 - (js_divergence / np.log(2))

    return s_maj, s_jsd


# -------------------------
# 3. Entropy score
# -------------------------
def s_ent(y_prob):
    N, K = y_prob.shape

    h = -np.sum(y_prob * np.log(y_prob + 1e-10), axis=1) / np.log(K)
    m = np.median(h)

    return 1 - min(abs(m - 0.5) / 0.5, 1.0)


# -------------------------
# 4. Drift score
# -------------------------
def s_drift(y_pred, ref_dist=None, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_pred))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)

    if ref_dist is None:
        ref_dist = np.ones(n_classes) / n_classes

    # jensenshannon trả về căn bậc hai của JSD (Distance)
    js_distance = jensenshannon(pred_dist, ref_dist)
    
    # Bình phương để có JSD (Divergence)
    js_divergence = js_distance**2
    
    return 1 - (js_divergence / np.log(2))


# -------------------------
# 5. Efficiency score
# -------------------------
def s_eff(mask):
    return np.mean(mask)


# -------------------------
# 6. Leakage score (generic probe)
# -------------------------
def s_leak_1(mask, y_true, probe_model, test_size=0.3, random_state=40):

    X_train, X_test, y_train, y_test = train_test_split(
        mask, y_true,
        test_size=test_size,
        random_state=random_state,
        stratify=y_true
    )

    probe_model.fit(X_train, y_train)

    if hasattr(probe_model, "predict_proba"):
        y_score = probe_model.predict_proba(X_test)
    else:
        y_score = probe_model.decision_function(X_test)

    auc = roc_auc_score(y_test, y_score, multi_class="ovr")
    return 1 - auc


# -------------------------
# 7. Logistic leakage
# -------------------------
def s_leak(X, y, test_size=0.3, random_state=40):

    Xtr, Xte, ytr, yte = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    probe = LogisticRegression(max_iter=1000)
    probe.fit(Xtr, ytr)

    y_score = probe.predict_proba(Xte)

    auc = roc_auc_score(yte, y_score, multi_class="ovr")
    return float(1 - auc)

# -------------------------
# 8. Final geometric mean score
# -------------------------
def S_san_plus(values):
    values = np.clip(values, 1e-10, 1)
    return np.prod(values) ** (1 / len(values))


In [2]:
def compute_ref_dist(y_train):
    y = y_train.astype(int).to_numpy()
    n_classes = len(np.unique(y))
    return np.bincount(y, minlength=n_classes) / len(y)

def run_sanity_evaluation(
    *,
    path_1,
    path_2,
    version_name,
    train_name,
    n_classes=3
):
    print(
        f"[START] Sanity evaluation | "
        f"train={train_name} | version={version_name}"
    )

    rows = []

    # -----------------
    # Load train
    # -----------------
    train_df = pd.read_csv(f"{path_1}/{train_name}.csv")
    y_train = train_df["label_3"]
    ref_dist = compute_ref_dist(y_train)

    # -----------------
    # Loop phases
    # -----------------
    for phase in [1, 2, 3, 4]:
        print(f"  -> Running phase {phase}...")

        # ---- probability matrix
        prob_df = pd.read_csv(
            f"{path_2}/probability_matrix_{version_name}_phase{phase}.csv"
        )

        y_true = prob_df["y_true"].astype(int).to_numpy()
        y_pred = prob_df["y_pred"].astype(int).to_numpy()
        y_prob = prob_df[
            ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
        ].to_numpy()

        # ---- test data (for leakage & efficiency)
        test_df = pd.read_csv(f"{path_1}/test_{phase}.csv")
        X_test = test_df.drop(columns=["label_3"], errors="ignore")
        mask = X_test.notna().astype(int).values

        # ---- metrics
        smaj, sjsd = s_maj_jsd(y_true, y_pred, n_classes=n_classes)

        snan   = s_nan(y_prob)
        sent   = s_ent(y_prob)
        sdrift = s_drift(y_pred, ref_dist, n_classes=n_classes)
        sleak  = s_leak(mask, y_true)
        s_eff_ = s_eff(mask)

        s_san_plus = S_san_plus([
            snan,
            sjsd,
            sent,
            sdrift,
            s_eff_,
            sleak
        ])

        rows.append({
            "train_name": train_name,
            "version_name": version_name,
            "phase": phase,
            "snan": snan,
            "smaj": smaj,
            "sjsd": sjsd,
            "sent": sent,
            "sdrift": sdrift,
            "sleak": sleak,
            "s_san+": s_san_plus,
            "s_eff": s_eff_
        })

    print(
        f"[END] Sanity evaluation DONE | "
        f"rows={len(rows)} | "
        f"phases=4"
    )

    columns_order = [
        "train_name",
        "version_name",
        "phase",
        "snan",
        "smaj",
        "sjsd",
        "sent",
        "sdrift",
        "sleak",
        "s_san+",
        "s_eff"
    ]

    return pd.DataFrame(rows)[columns_order]


# Chạy với các version data

## V1 (Median)

In [3]:
path_1 = "/kaggle/input/lo-dataset/Median/Median"

In [4]:
path_2 = "/kaggle/input/datasets/anhtran10/gru-results/GRU_results"

In [5]:
df_v1 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V1 (Median)",
    train_name="train_median"
)
df_v1

[START] Sanity evaluation | train=train_median | version=V1 (Median)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median,V1 (Median),1,1,0.005950,0.999684,0.004291,0.999684,0.5,0.359097,1.0
1,train_median,V1 (Median),2,1,0.006163,0.999821,0.003546,0.999822,0.5,0.347884,1.0
2,train_median,V1 (Median),3,1,0.004988,0.999981,0.001150,0.999981,0.5,0.288354,1.0
3,train_median,V1 (Median),4,1,0.004795,0.999990,0.000011,0.999990,0.5,0.132481,1.0


In [6]:
df_v1.to_csv("results_v1.csv", index=False)

## V2 (Median CDSMOTE)

In [7]:
df_v2 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V2 (Median CDSMOTE)",
    train_name="train_median_cdsmote"
)
df_v2

[START] Sanity evaluation | train=train_median_cdsmote | version=V2 (Median CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_cdsmote,V2 (Median CDSMOTE),1,1,0.002491,0.999625,0.000003,0.548354,0.5,0.097491,1.0
1,train_median_cdsmote,V2 (Median CDSMOTE),2,1,0.002207,0.999657,0.000001,0.547858,0.5,0.084083,1.0
2,train_median_cdsmote,V2 (Median CDSMOTE),3,1,0.004575,0.999969,0.000003,0.554170,0.5,0.094205,1.0
3,train_median_cdsmote,V2 (Median CDSMOTE),4,1,0.010893,0.999531,0.000006,0.567781,0.5,0.109174,1.0


In [8]:
df_v2.to_csv("results_v2.csv", index=False)

## V3 (Median SASMOTE)

In [9]:
df_v3 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V3 (Median SASMOTE)",
    train_name="train_median_sasmote"
)
df_v3

[START] Sanity evaluation | train=train_median_sasmote | version=V3 (Median SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_sasmote,V3 (Median SASMOTE),1,1,0.001258,0.999345,2.607302e-06,0.545259,0.5,0.094460,1.0
1,train_median_sasmote,V3 (Median SASMOTE),2,1,0.001091,0.999175,3.674809e-07,0.544797,0.5,0.068131,1.0
2,train_median_sasmote,V3 (Median SASMOTE),3,1,0.004975,0.999903,8.989033e-06,0.555240,0.5,0.116463,1.0
3,train_median_sasmote,V3 (Median SASMOTE),4,1,0.009912,0.999658,1.408499e-05,0.565900,0.5,0.125908,1.0


In [10]:
df_v3.to_csv("results_v3.csv", index=False)

## V4 (Median RadiusSMOTE)

In [11]:
df_v4 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V4 (Median RadiusSMOTE)",
    train_name="train_median_radiussmote"
)
df_v4

[START] Sanity evaluation | train=train_median_radiussmote | version=V4 (Median RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_median_radiussmote,V4 (Median RadiusSMOTE),1,1,0.002097,0.999617,0.000109,0.547772,0.5,0.176045,1.0
1,train_median_radiussmote,V4 (Median RadiusSMOTE),2,1,0.002626,0.999772,0.000026,0.549117,0.5,0.138846,1.0
2,train_median_radiussmote,V4 (Median RadiusSMOTE),3,1,0.005466,0.999984,0.001074,0.556288,0.5,0.258552,1.0
3,train_median_radiussmote,V4 (Median RadiusSMOTE),4,1,0.007382,0.999909,0.000139,0.560602,0.5,0.184100,1.0


In [12]:
df_v4.to_csv("results_v4.csv", index=False)

## V5 (Mean)

In [13]:
path_1 = "/kaggle/input/lo-dataset/Mean/Mean"

In [14]:
df_v5 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V5 (Mean)",
    train_name="train_mean"
)
df_v5

[START] Sanity evaluation | train=train_mean | version=V5 (Mean)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean,V5 (Mean),1,1,0.009873,0.999039,0.013819,0.999039,0.5,0.436282,1.0
1,train_mean,V5 (Mean),2,1,0.005866,0.999910,0.009803,0.999910,0.5,0.412135,1.0
2,train_mean,V5 (Mean),3,1,0.009996,0.999263,0.013475,0.999263,0.5,0.434487,1.0
3,train_mean,V5 (Mean),4,1,0.004930,0.999996,0.000098,0.999996,0.5,0.191447,1.0


In [15]:
df_v5.to_csv("results_v5.csv", index=False)

## V6 (Mean CDSMOTE)

In [16]:
df_v6 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V6 (Mean CDSMOTE)",
    train_name="train_mean_cdsmote"
)
df_v6

[START] Sanity evaluation | train=train_mean_cdsmote | version=V6 (Mean CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_cdsmote,V6 (Mean CDSMOTE),1,1,0.010047,0.999055,0.000033,0.566215,0.5,0.145172,1.0
1,train_mean_cdsmote,V6 (Mean CDSMOTE),2,1,0.007427,0.999532,0.000010,0.560810,0.5,0.117863,1.0
2,train_mean_cdsmote,V6 (Mean CDSMOTE),3,1,0.008486,0.999654,0.000013,0.563236,0.5,0.124671,1.0
3,train_mean_cdsmote,V6 (Mean CDSMOTE),4,1,0.009725,0.999683,0.000006,0.565493,0.5,0.109709,1.0


In [17]:
df_v6.to_csv("results_v6.csv", index=False)

## V7 (Mean SASMOTE)

In [18]:
df_v7 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V7 (Mean SASMOTE)",
    train_name="train_mean_sasmote"
)
df_v7

[START] Sanity evaluation | train=train_mean_sasmote | version=V7 (Mean SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_sasmote,V7 (Mean SASMOTE),1,1,0.012615,0.998952,0.000350,0.571691,0.5,0.215445,1.0
1,train_mean_sasmote,V7 (Mean SASMOTE),2,1,0.012299,0.999005,0.000173,0.571068,0.5,0.191442,1.0
2,train_mean_sasmote,V7 (Mean SASMOTE),3,1,0.010602,0.999262,0.000055,0.567645,0.5,0.157916,1.0
3,train_mean_sasmote,V7 (Mean SASMOTE),4,1,0.011525,0.999439,0.000040,0.568900,0.5,0.149995,1.0


In [19]:
df_v7.to_csv("results_v7.csv", index=False)

## V8 (Mean RadiusSMOTE)

In [20]:
df_v8 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V8 (Mean RadiusSMOTE)",
    train_name="train_mean_radiussmote"
)
df_v8

[START] Sanity evaluation | train=train_mean_radiussmote | version=V8 (Mean RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_mean_radiussmote,V8 (Mean RadiusSMOTE),1,1,0.013958,0.998251,0.002788,0.573685,0.5,0.304568,1.0
1,train_mean_radiussmote,V8 (Mean RadiusSMOTE),2,1,0.012764,0.998790,0.002974,0.571856,0.5,0.307745,1.0
2,train_mean_radiussmote,V8 (Mean RadiusSMOTE),3,1,0.006679,0.999900,0.000509,0.559186,0.5,0.228520,1.0
3,train_mean_radiussmote,V8 (Mean RadiusSMOTE),4,1,0.008279,0.999846,0.000046,0.562383,0.5,0.153157,1.0


In [21]:
df_v8.to_csv("results_v8.csv", index=False)

## V9 (Extra Trees)

In [22]:
path_1 = "/kaggle/input/lo-dataset/Extra_trees/Extra_trees"

In [23]:
df_v9 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V9 (MissForest)",
    train_name="train_extra"
)
df_v9

[START] Sanity evaluation | train=train_extra | version=V9 (MissForest)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra,V9 (MissForest),1,1,0.053875,0.988649,0.022094,0.988648,0.5,0.470134,1.0
1,train_extra,V9 (MissForest),2,1,0.027289,0.996030,0.005190,0.996029,0.5,0.370207,1.0
2,train_extra,V9 (MissForest),3,1,0.037904,0.992944,0.003334,0.992943,0.5,0.343521,1.0
3,train_extra,V9 (MissForest),4,1,0.004917,0.999995,0.000001,0.999995,0.5,0.092231,1.0


In [24]:
df_v9.to_csv("results_v9.csv", index=False)

## V10 (Extra Trees CDSMOTE)

In [25]:
df_v10 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V10 (MissForest CDSMOTE)",
    train_name="train_extra_cdsmote"
)
df_v10

[START] Sanity evaluation | train=train_extra_cdsmote | version=V10 (MissForest CDSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_cdsmote,V10 (MissForest CDSMOTE),1,1,0.006814,0.999153,1.777513e-05,0.558989,0.5,0.130608,1.0
1,train_extra_cdsmote,V10 (MissForest CDSMOTE),2,1,0.007408,0.999918,1.254898e-05,0.560429,0.5,0.123314,1.0
2,train_extra_cdsmote,V10 (MissForest CDSMOTE),3,1,0.009950,0.999652,5.608789e-06,0.565601,0.5,0.107986,1.0
3,train_extra_cdsmote,V10 (MissForest CDSMOTE),4,1,0.009492,0.999716,6.583410e-08,0.564911,0.5,0.051469,1.0


In [26]:
df_v10.to_csv("results_v10.csv", index=False)

## V11 (Extra Trees SASMOTE)

In [27]:
df_v11 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V11 (MissForest SASMOTE)",
    train_name="train_extra_sasmote"
)
df_v11

[START] Sanity evaluation | train=train_extra_sasmote | version=V11 (MissForest SASMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_sasmote,V11 (MissForest SASMOTE),1,1,0.001207,0.998836,2.542974e-06,0.544893,0.5,0.094049,1.0
1,train_extra_sasmote,V11 (MissForest SASMOTE),2,1,0.001975,0.999327,2.315179e-06,0.547354,0.5,0.092666,1.0
2,train_extra_sasmote,V11 (MissForest SASMOTE),3,1,0.005892,0.999775,2.657005e-06,0.557411,0.5,0.095113,1.0
3,train_extra_sasmote,V11 (MissForest SASMOTE),4,1,0.008408,0.999825,3.288466e-08,0.562793,0.5,0.045818,1.0


In [28]:
df_v11.to_csv("results_v11.csv", index=False)

## V12 (Extra Trees RadiusSMOTE)

In [29]:
df_v12 = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="V12 (MissForest RadiusSMOTE)",
    train_name="train_extra_radiussmote"
)
df_v12

[START] Sanity evaluation | train=train_extra_radiussmote | version=V12 (MissForest RadiusSMOTE)
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),1,1,0.011648,0.999189,0.012035,0.569805,0.5,0.388269,1.0
1,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),2,1,0.009937,0.999455,0.002791,0.566312,0.5,0.304039,1.0
2,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),3,1,0.014171,0.998803,0.002809,0.574752,0.5,0.305078,1.0
3,train_extra_radiussmote,V12 (MissForest RadiusSMOTE),4,1,0.007285,0.999918,0.000011,0.560376,0.5,0.120007,1.0


In [30]:
df_v12.to_csv("results_v12.csv", index=False)